In [1]:
import torch
from torch import nn
from transformers import T5ForConditionalGeneration, T5Tokenizer
from typing import Dict, List, Optional
from tqdm import tqdm

def preprocess_dataset(input_texts: List[str], target_texts: List[str], max_words: int = 100):
    """Preprocess and filter the dataset."""
    processed_inputs = []
    processed_targets = []
    
    for inp, tgt in zip(input_texts, target_texts):
        # Convert to string and strip whitespace
        inp = str(inp).strip()
        tgt = str(tgt).strip()
        
        # Skip empty pairs
        if not inp or not tgt:
            continue
            
        # Truncate to max_words
        inp_words = inp.split()[:max_words]
        tgt_words = tgt.split()[:max_words]
        
        # Rejoin words
        processed_inputs.append(' '.join(inp_words))
        processed_targets.append(' '.join(tgt_words))
    
    return processed_inputs, processed_targets

class DialogueModel(nn.Module):
    def __init__(
        self,
        model_name: str = "t5-base",
        max_length: int = 128,  # Reduced from 512 to handle memory better
        device: str = "cuda" if torch.cuda.is_available() else "cpu"
    ):
        super().__init__()
        self.model_name = model_name
        self.max_length = max_length
        self.device = device
        
        # Initialize the T5 model and tokenizer
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.tokenizer = T5Tokenizer.from_pretrained(model_name, model_max_length=max_length)
        self.model.to(device)
        
    def forward(
        self,
        input_text: List[str],
        labels: Optional[List[str]] = None
    ) -> Dict[str, torch.Tensor]:
        # Tokenize inputs
        inputs = self.tokenizer(
            input_text,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)
        
        if labels is not None:
            # Tokenize labels
            with torch.no_grad():
                label_tokens = self.tokenizer(
                    labels,
                    padding=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt"
                ).input_ids.to(self.device)
            
            # Replace padding token id with -100
            label_tokens[label_tokens == self.tokenizer.pad_token_id] = -100
            
            # Forward pass with labels
            outputs = self.model(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                labels=label_tokens
            )
            
            return {"loss": outputs.loss, "logits": outputs.logits}
        else:
            # Generation mode
            outputs = self.model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_length=self.max_length,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True
            )
            return {"generated": outputs}

    def train(
        self,
        input_texts: List[str],
        target_texts: List[str],
        val_input_texts: Optional[List[str]] = None,
        val_target_texts: Optional[List[str]] = None,
        batch_size: int = 8,
        epochs: int = 3,
        learning_rate: float = 2e-5,
    ):
        """Train the model on the provided data."""
        # Preprocess the data
        print("Preprocessing training data...")
        input_texts, target_texts = preprocess_dataset(input_texts, target_texts)
        if val_input_texts is not None and val_target_texts is not None:
            val_input_texts, val_target_texts = preprocess_dataset(val_input_texts, val_target_texts)
        
        if len(input_texts) == 0:
            raise ValueError("No valid training examples after preprocessing!")
            
        # Print data statistics
        input_lengths = [len(text.split()) for text in input_texts]
        target_lengths = [len(text.split()) for text in target_texts]
        print(f"Processed dataset statistics:")
        print(f"Number of training examples: {len(input_texts)}")
        print(f"Input length stats: min={min(input_lengths)}, max={max(input_lengths)}, avg={sum(input_lengths)/len(input_lengths):.2f}")
        print(f"Target length stats: min={min(target_lengths)}, max={max(target_lengths)}, avg={sum(target_lengths)/len(target_lengths):.2f}")
        
        # Set model to training mode
        self.model.train()
        
        # Initialize optimizer
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate)
        
        # Training loop
        for epoch in range(epochs):
            total_loss = 0
            valid_batches = 0
            
            # Process in batches
            progress_bar = tqdm(range(0, len(input_texts), batch_size), desc=f"Epoch {epoch+1}/{epochs}")
            for i in progress_bar:
                batch_inputs = input_texts[i:i + batch_size]
                batch_targets = target_texts[i:i + batch_size]
                
                # Clear gradients
                optimizer.zero_grad()
                
                try:
                    # Forward pass
                    outputs = self.forward(
                        input_text=batch_inputs,
                        labels=batch_targets
                    )
                    
                    loss = outputs["loss"]
                    total_loss += loss.item()
                    valid_batches += 1
                    
                    # Update progress bar
                    progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
                    
                    # Backward pass and optimization
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    
                except RuntimeError as e:
                    print(f"\nError in batch {i//batch_size}: {str(e)}")
                    continue
            
            # Print epoch statistics
            if valid_batches > 0:
                avg_loss = total_loss / valid_batches
                print(f"\nEpoch {epoch + 1}/{epochs}, Average Loss: {avg_loss:.4f}")
            
            # Validation step
            if val_input_texts and val_target_texts:
                self._validate(val_input_texts, val_target_texts, batch_size)
    
    def _validate(self, val_inputs, val_targets, batch_size):
        self.model.eval()
        total_val_loss = 0
        valid_batches = 0
        
        with torch.no_grad():
            for i in range(0, len(val_inputs), batch_size):
                try:
                    batch_val_inputs = val_inputs[i:i + batch_size]
                    batch_val_targets = val_targets[i:i + batch_size]
                    
                    val_outputs = self.forward(
                        input_text=batch_val_inputs,
                        labels=batch_val_targets
                    )
                    total_val_loss += val_outputs["loss"].item()
                    valid_batches += 1
                except RuntimeError as e:
                    print(f"Error in validation batch {i//batch_size}: {str(e)}")
                    continue
        
        if valid_batches > 0:
            avg_val_loss = total_val_loss / valid_batches
            print(f"Validation Loss: {avg_val_loss:.4f}")
        
        self.model.train()

In [ ]:
pip install nltk rouge-score bert-score numpy

In [2]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score
import numpy as np
from typing import List, Dict, Union
import re
from collections import Counter

class ResponseEvaluator:
    def __init__(self):
        # Initialize ROUGE scorer
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        # BLEU smoothing function
        self.smoother = SmoothingFunction().method1
        
    def _preprocess_text(self, text: str) -> str:
        """Clean and normalize text for evaluation."""
        if not isinstance(text, str):
            text = str(text)
        # Convert to lowercase
        text = text.lower()
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove special characters but keep punctuation
        text = re.sub(r'[^a-z0-9\s.,!?\'"]', '', text)
        return text.strip()
    
    def _calculate_lexical_diversity(self, text: str) -> float:
        """Calculate type-token ratio for lexical diversity."""
        words = text.split()
        if not words:
            return 0.0
        return len(set(words)) / len(words)
    
    def _calculate_response_coherence(self, text: str) -> Dict[str, Union[bool, int]]:
        """Analyze response coherence based on basic linguistic features."""
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        
        return {
            'sentence_count': len(sentences),
            'avg_sentence_length': np.mean([len(s.split()) for s in sentences]) if sentences else 0,
            'has_discourse_markers': any(marker in text.lower() for marker in 
                ['however', 'therefore', 'because', 'although', 'furthermore', 'moreover'])
        }
    
    def _calculate_engagement_metrics(self, text: str) -> Dict[str, bool]:
        """Analyze engagement features in the response."""
        return {
            'has_question': '?' in text,
            'has_exclamation': '!' in text,
            'has_personal_pronouns': any(pronoun in text.lower().split() for pronoun in 
                ['i', 'you', 'we', 'our', 'your']),
        }

    def evaluate_responses(self, 
                         generated_responses: List[str], 
                         reference_responses: List[str]) -> Dict[str, float]:
        """
        Evaluate generated responses using multiple metrics.
        
        Args:
            generated_responses: List of generated responses
            reference_responses: List of reference/ground truth responses
            
        Returns:
            Dictionary containing various evaluation metrics
        """
        if len(generated_responses) != len(reference_responses):
            raise ValueError("Number of generated and reference responses must match")
            
        results = {
            'bleu': [],
            'rouge': {'rouge1': [], 'rouge2': [], 'rougeL': []},
            'bert_score': {'precision': [], 'recall': [], 'f1': []},
            'lexical_diversity': [],
            'coherence': {'sentence_count': [], 'avg_sentence_length': []},
            'engagement': {'question_rate': [], 'exclamation_rate': [], 'personal_pronouns_rate': []}
        }
        
        for gen, ref in zip(generated_responses, reference_responses):
            # Preprocess texts
            gen_clean = self._preprocess_text(gen)
            ref_clean = self._preprocess_text(ref)
            
            if not gen_clean or not ref_clean:
                continue
                
            # BLEU score
            try:
                bleu = sentence_bleu([ref_clean.split()], gen_clean.split(), 
                                   smoothing_function=self.smoother)
                results['bleu'].append(bleu)
            except Exception as e:
                print(f"Error calculating BLEU score: {e}")
                
            # ROUGE scores
            try:
                rouge_scores = self.rouge_scorer.score(ref_clean, gen_clean)
                results['rouge']['rouge1'].append(rouge_scores['rouge1'].fmeasure)
                results['rouge']['rouge2'].append(rouge_scores['rouge2'].fmeasure)
                results['rouge']['rougeL'].append(rouge_scores['rougeL'].fmeasure)
            except Exception as e:
                print(f"Error calculating ROUGE scores: {e}")
                
            # BERTScore
            try:
                P, R, F1 = score([gen_clean], [ref_clean], lang='en', verbose=False)
                results['bert_score']['precision'].append(P.item())
                results['bert_score']['recall'].append(R.item())
                results['bert_score']['f1'].append(F1.item())
            except Exception as e:
                print(f"Error calculating BERTScore: {e}")
                
            # Additional metrics
            results['lexical_diversity'].append(self._calculate_lexical_diversity(gen_clean))
            
            coherence_metrics = self._calculate_response_coherence(gen_clean)
            results['coherence']['sentence_count'].append(coherence_metrics['sentence_count'])
            results['coherence']['avg_sentence_length'].append(coherence_metrics['avg_sentence_length'])
            
            engagement_metrics = self._calculate_engagement_metrics(gen_clean)
            results['engagement']['question_rate'].append(int(engagement_metrics['has_question']))
            results['engagement']['exclamation_rate'].append(int(engagement_metrics['has_exclamation']))
            results['engagement']['personal_pronouns_rate'].append(int(engagement_metrics['has_personal_pronouns']))
        
        # Calculate final aggregated scores
        final_results = {
            'bleu': np.mean(results['bleu']) if results['bleu'] else 0.0,
            'rouge1_f1': np.mean(results['rouge']['rouge1']) if results['rouge']['rouge1'] else 0.0,
            'rouge2_f1': np.mean(results['rouge']['rouge2']) if results['rouge']['rouge2'] else 0.0,
            'rougeL_f1': np.mean(results['rouge']['rougeL']) if results['rouge']['rougeL'] else 0.0,
            'bert_score_f1': np.mean(results['bert_score']['f1']) if results['bert_score']['f1'] else 0.0,
            'lexical_diversity': np.mean(results['lexical_diversity']) if results['lexical_diversity'] else 0.0,
            'avg_sentence_count': np.mean(results['coherence']['sentence_count']) if results['coherence']['sentence_count'] else 0.0,
            'avg_sentence_length': np.mean(results['coherence']['avg_sentence_length']) if results['coherence']['avg_sentence_length'] else 0.0,
            'question_rate': np.mean(results['engagement']['question_rate']) if results['engagement']['question_rate'] else 0.0,
            'exclamation_rate': np.mean(results['engagement']['exclamation_rate']) if results['engagement']['exclamation_rate'] else 0.0,
            'personal_pronouns_rate': np.mean(results['engagement']['personal_pronouns_rate']) if results['engagement']['personal_pronouns_rate'] else 0.0
        }
        
        return final_results

    def qualitative_analysis(self, response: str) -> Dict[str, Union[int, float, bool]]:
        """
        Perform detailed qualitative analysis of a single response.
        
        Args:
            response: Single response to analyze
            
        Returns:
            Dictionary containing qualitative metrics
        """
        clean_response = self._preprocess_text(response)
        
        # Basic statistics
        word_count = len(clean_response.split())
        char_count = len(clean_response)
        
        # Combine all metrics
        analysis = {
            'length': {
                'word_count': word_count,
                'char_count': char_count,
                'avg_word_length': char_count / word_count if word_count > 0 else 0
            },
            'lexical_diversity': self._calculate_lexical_diversity(clean_response),
            'coherence': self._calculate_response_coherence(clean_response),
            'engagement': self._calculate_engagement_metrics(clean_response)
        }
        
        return analysis

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from tqdm import tqdm
# from model import DialogueModel
# from evaluation import evaluate_responses, qualitative_analysis

# Load the dataset
df = pd.read_csv('/kaggle/input/nlp1112/dialoconan.csv')

# Display basic information about the dataset
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

# Analyze dialogue structure
print("\nDialogue Structure Analysis:")
print("Number of unique dialogues:", df['dialogue_id'].nunique())

# Analyze turns in conversations
turns_per_dialogue = df.groupby('dialogue_id').size()
print("\nTurns per conversation:")
print(turns_per_dialogue.value_counts().sort_index())
print(f"Average turns per conversation: {turns_per_dialogue.mean():.2f}")

# Analyze distribution of turn types
turn_types = df['type'].value_counts()
print("\nDistribution of Turn Types (HS vs CN):")
print(turn_types)

# Analyze hate speech categories
hate_targets = df[df['type'] == 'HS']['TARGET'].value_counts()
print("\nHate Speech Target Categories:")
print(hate_targets)

# Analyze text lengths
df['text_length'] = df['text'].str.len()
print("\nText Length Statistics:")
print(df.groupby('type')['text_length'].describe())

# Visualize hate categories distribution
plt.figure(figsize=(10, 6))
hate_targets.plot(kind='bar')
plt.title('Distribution of Hate Speech Target Categories')
plt.xlabel('Target Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Split into train, validation, and test sets (80-10-10 split)
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temp
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
# Second split: 10% validation, 10% test (half of the remaining 20%)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("\nDataset Splits:")
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

# Initialize the model


# Prepare training data
train_inputs = train_df[train_df['type'] == 'HS']['text'].tolist()
train_targets = train_df[train_df['type'] == 'CN']['text'].tolist()
val_inputs = val_df[val_df['type'] == 'HS']['text'].tolist()
val_targets = val_df[val_df['type'] == 'CN']['text'].tolist()

# Train the model
print("\nTraining the model...")
model = DialogueModel(model_name="t5-base")
model.train(
    input_texts=train_inputs,
    target_texts=train_targets,
    val_input_texts=val_inputs,
    val_target_texts=val_targets,
    batch_size=8,
    epochs=3
)



In [14]:
def prepare_test_pairs(test_df):
    """Prepare matched pairs of HS and CN from test set."""
    # Get unique dialogue IDs
    dialogue_ids = test_df['dialogue_id'].unique()
    
    paired_inputs = []
    paired_references = []
    
    for dialog_id in dialogue_ids:
        # Get all messages in this dialogue
        dialog = test_df[test_df['dialogue_id'] == dialog_id].sort_values('index')
        
        # Find HS-CN pairs in the dialogue
        hs_messages = dialog[dialog['type'] == 'HS']
        cn_messages = dialog[dialog['type'] == 'CN']
        
        # Match each HS with its corresponding CN
        for _, hs_row in hs_messages.iterrows():
            # Find the CN that follows this HS
            following_cn = cn_messages[cn_messages['index'] > hs_row['index']].iloc[0] if len(cn_messages[cn_messages['index'] > hs_row['index']]) > 0 else None
            
            if following_cn is not None:
                paired_inputs.append(hs_row['text'])
                paired_references.append(following_cn['text'])
    
    return paired_inputs, paired_references

def generate_and_evaluate_responses(model, test_df, batch_size=8):
    """Generate and evaluate responses for the test set."""
    # Prepare paired test data
    print("Preparing test pairs...")
    test_inputs, reference_responses = prepare_test_pairs(test_df)
    print(f"Number of test pairs: {len(test_inputs)}")
    
    # Generate responses
    print("\nGenerating responses...")
    generated_responses = []
    
    for i in tqdm(range(0, len(test_inputs), batch_size), desc="Generating Responses", unit="batch"):
        batch_inputs = test_inputs[i:min(i + batch_size, len(test_inputs))]
        
        # Tokenize inputs
        batch_encoded = model.tokenizer(
            batch_inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=model.max_length
        ).to(model.device)
        
        # Generate responses
        with torch.no_grad():
            batch_outputs = model.model.generate(
                input_ids=batch_encoded.input_ids,
                attention_mask=batch_encoded.attention_mask,
                max_length=model.max_length,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True
            )
            
        # Decode responses
        batch_responses = model.tokenizer.batch_decode(batch_outputs, skip_special_tokens=True)
        generated_responses.extend(batch_responses)
    
    # Initialize evaluator and compute scores
    print("\nEvaluating responses...")
    evaluator = ResponseEvaluator()
    evaluation_scores = evaluator.evaluate_responses(generated_responses, reference_responses)
    
    # Print quantitative results
    print("\nQuantitative Evaluation Scores:")
    for metric, score in evaluation_scores.items():
        print(f"{metric}: {score:.4f}")
    
    # Print qualitative analysis for a sample
    print("\nQualitative Analysis Examples:")
    for i, response in enumerate(generated_responses[:5]):
        print(f"\nResponse {i+1}:")
        print(f"Input: {test_inputs[i]}")
        print(f"Generated: {response}")
        print(f"Reference: {reference_responses[i]}")
        qual_metrics = evaluator.qualitative_analysis(response)
        print("Qualitative Metrics:")
        for category, metrics in qual_metrics.items():
            print(f"\n{category.capitalize()}:")
            if isinstance(metrics, dict):
                for k, v in metrics.items():
                    print(f"  {k}: {v}")
            else:
                print(f"  {metrics}")
    
    return generated_responses, reference_responses, evaluation_scores

In [ ]:
# Generate and evaluate responses
generated_responses, reference_responses, evaluation_scores = generate_and_evaluate_responses(model, test_df)

In [ ]:
print(response)